In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3


Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires 

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: True
Visible GPUs: 2
GPU name: Tesla T4


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import transformers

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    logging,
    DataCollatorWithPadding
)

# ---------------------------
# CLEANUP
# ---------------------------
path = "/kaggle/working/state.db"
if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")

# ---------------------------
# METRICS (argmax-based during training eval)
# We'll do *threshold tuning* separately after training.
# ---------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    f1 = f1_score(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="binary", zero_division=0)
    rec = recall_score(labels, preds, average="binary", zero_division=0)

    return {
        "f1": f1,
        "accuracy": acc,
        "precision": prec,
        "recall": rec
    }

# ---------------------------
# LOAD DATA
# ---------------------------
base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A"

training_path = base_path + "/train.parquet"
validation_path = base_path + "/validation.parquet"
test_sample_path = base_path + "/test_sample.parquet"
test_path = base_path + "/test.parquet"
sample_sub_path = base_path + "/sample_submission.csv"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_sample_df = pd.read_parquet(test_sample_path)
test_df = pd.read_parquet(test_path)
sample_sub_df = pd.read_csv(sample_sub_path)

for df in [training_df, validation_df, test_sample_df, test_df]:
    if "__index_level_0__" in df.columns:
        df.drop(columns=["__index_level_0__"], inplace=True)

print("Loaded shapes:")
print("train:", training_df.shape)
print("val:", validation_df.shape)
print("test:", test_df.shape)
print("test_sample:", test_sample_df.shape)
print("sample_submission:", sample_sub_df.shape)

# Optional: labeled subset of test (if test_sample has labels)
test_sample_merged = pd.merge(test_df, test_sample_df, on="code", how="inner")
print("test_sample_merged:", test_sample_merged.shape)

# ---------------------------
# TOKENIZER / PREPROCESS
# ---------------------------
pretrained_model = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

MAX_LEN = 512  # <-- big change vs 256; often helps for code
def preprocess_function(examples):
    return tokenizer(examples["code"], truncation=True, max_length=MAX_LEN)

training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)
test_sample_merged_dataset = Dataset.from_pandas(test_sample_merged)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True, remove_columns=[c for c in training_dataset.column_names if c not in ["code", "label", "ID"]])
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True, remove_columns=[c for c in validation_dataset.column_names if c not in ["code", "label", "ID"]])
test_tokenized_set = test_dataset.map(preprocess_function, batched=True, remove_columns=[c for c in test_dataset.column_names if c not in ["code", "ID"]])
test_sample_merged_tokenized_set = test_sample_merged_dataset.map(preprocess_function, batched=True, remove_columns=[c for c in test_sample_merged_dataset.column_names if c not in ["code", "label", "ID"]])

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")
test_tokenized_set.set_format("torch")
test_sample_merged_tokenized_set.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

# ---------------------------
# MODEL
# ---------------------------
model = AutoModelForSequenceClassification.from_pretrained(pretrained_model, num_labels=2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("\nRunning on device:", device)

logging.set_verbosity_info()

# ---------------------------
# CLASS WEIGHTS (imbalance handling)
# ---------------------------
# Works if training_df has 'label' column with 0/1.
label_counts = training_df["label"].value_counts().sort_index()
# Avoid division by zero; also keep float32 weights.
# Weight for class i = total / (2 * count_i)  (simple balanced heuristic)
total = float(label_counts.sum())
w0 = total / (2.0 * float(label_counts.get(0, 1)))
w1 = total / (2.0 * float(label_counts.get(1, 1)))
class_weights = torch.tensor([w0, w1], dtype=torch.float32)
print("Label counts:", label_counts.to_dict())
print("Class weights:", class_weights.tolist())

# Custom Trainer with weighted CE loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.get("logits")

        if self.class_weights is not None:
            cw = self.class_weights.to(logits.device)
            loss = torch.nn.functional.cross_entropy(logits, labels, weight=cw)
        else:
            loss = torch.nn.functional.cross_entropy(logits, labels)

        return (loss, outputs) if return_outputs else loss

# ---------------------------
# TRAINING ARGS
# ---------------------------
OUTPUT_DIR = "/kaggle/working/checkpoints_task_a"
LOG_DIR = "/kaggle/working/logs_task_a"

# Keep runtime similar by staying near your original settings.
# If you see OOM due to max_length=512, reduce batch_size to 4 or set gradient_accumulation_steps higher.
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    seed=42,

    learning_rate=2e-5,              # slightly lower LR often helps stability/generalization
    per_device_train_batch_size=8,   # adjust if OOM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,   # effective batch ~16

    num_train_epochs=3,              # same as you had; early stopping will cut if needed
    weight_decay=0.01,
    warmup_ratio=0.1,

    lr_scheduler_type="cosine",      # change from default linear
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    metric_for_best_model="f1",
    greater_is_better=True,
    load_best_model_at_end=True,

    logging_dir=LOG_DIR,
    logging_steps=500,

    fp16=True,
    disable_tqdm=False,
    dataloader_num_workers=0,
    report_to=[]
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# ---------------------------
# RESUME IF CHECKPOINT
# ---------------------------
resume = False
last_ckpt = None
if os.path.isdir(OUTPUT_DIR):
    from transformers.trainer_utils import get_last_checkpoint
    last_ckpt = get_last_checkpoint(OUTPUT_DIR)
    if last_ckpt is not None:
        resume = True
        print("Found checkpoint:", last_ckpt)

if resume:
    trainer.train(resume_from_checkpoint=last_ckpt)
else:
    trainer.train()

print("Eval on validation (argmax):", trainer.evaluate(validation_tokenized_set))
if len(test_sample_merged_tokenized_set) > 0 and "labels" in test_sample_merged_tokenized_set.features:
    print("Eval on test_sample_merged (argmax):", trainer.evaluate(test_sample_merged_tokenized_set))

# ---------------------------
# THRESHOLD TUNING ON VALIDATION
# ---------------------------
def best_threshold_from_logits(logits: np.ndarray, labels: np.ndarray):
    # Convert logits -> prob of class 1
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    best_t, best_f1 = 0.5, -1.0

    for t in np.linspace(0.05, 0.95, 19):  # coarse grid
        preds = (probs >= t).astype(int)
        f1 = f1_score(labels, preds, average="binary", zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)

    return best_t, best_f1

val_pred = trainer.predict(validation_tokenized_set)
val_logits = val_pred.predictions
val_labels = val_pred.label_ids

t_star, f1_star = best_threshold_from_logits(val_logits, val_labels)
print(f"\nBest threshold on validation: {t_star:.2f}  -> F1: {f1_star:.4f}")

# ---------------------------
# PREDICT TEST WITH THRESHOLD (NOT ARGMAX)
# ---------------------------
test_pred = trainer.predict(test_tokenized_set)
test_logits = test_pred.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()[:, 1]
predicted_labels = (test_probs >= t_star).astype(int)

pred_df = pd.DataFrame({
    "ID": test_df["ID"].values,
    "label": predicted_labels
})

submission_df = sample_sub_df[["ID"]].merge(pred_df, on="ID", how="left")
missing = submission_df["label"].isna().sum()
print("Missing labels after merge:", missing)
submission_df["label"] = submission_df["label"].fillna(0).astype(int)

out_path = "/kaggle/working/submission.csv"
submission_df.to_csv(out_path, index=False)
print("Saved:", out_path)
print(submission_df.head())


state.db deleted
Loaded shapes:
train: (500000, 4)
val: (100000, 4)
test: (500000, 2)
test_sample: (1000, 4)
sample_submission: (1000, 2)
test_sample_merged: (437, 5)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

...


Running on device: cuda
Label counts: {0: 238475, 1: 261525}
Class weights: [1.048327922821045, 0.955931544303894]


Loading model from /kaggle/working/checkpoints_task_a/checkpoint-31250.


Found checkpoint: /kaggle/working/checkpoints_task_a/checkpoint-31250


The following columns in the Training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: code. If code are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 500,000
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Training with DataParallel so batch size has been adjusted to: 16
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 2
  Total optimization steps = 46,875
  Number of trainable parameters = 125,931,266
  Continuing training from checkpoint, will skip to saved global_step
  Continuing training from epoch 2
  Continuing training from global step 31250
  Will skip the first 2 epochs then the first 0 batches in the first epoch.


Epoch,Training Loss,Validation Loss


In [2]:
import torch
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


cuda available: False


In [1]:
!ls -lh /kaggle/working | grep -E "submission|checkpoints" || true


drwxr-xr-x 4 root root 4.0K Jan 19 07:11 checkpoints_task_a
-rw-r--r-- 1 root root 8.8K Jan 19 07:11 submission.csv


In [2]:
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification

# ---------------------------
# LOAD BEST MODEL
# ---------------------------
BEST_CKPT = "/kaggle/working/checkpoints_task_a/checkpoint-46875"

model = AutoModelForSequenceClassification.from_pretrained(BEST_CKPT)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

print("Loaded model from:", BEST_CKPT)
print("Device:", device)

# ---------------------------
# RUN PREDICTION ON FULL TEST
# ---------------------------
test_pred = trainer.predict(test_tokenized_set)

logits = test_pred.predictions
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

# threshold from validation
t_star = 0.45
labels = (probs >= t_star).astype(int)

# ---------------------------
# BUILD FULL SUBMISSION (500k rows)
# ---------------------------
submission_df = pd.DataFrame({
    "ID": test_df["ID"].values,
    "label": labels
})

print("Submission rows:", submission_df.shape[0])
print("Label distribution:\n", submission_df["label"].value_counts())

out_path = "/kaggle/working/submission_full.csv"
submission_df.to_csv(out_path, index=False)

print("Saved:", out_path)
submission_df.head()


Loaded model from: /kaggle/working/checkpoints_task_a/checkpoint-46875
Device: cuda


Submission rows: 500000
Label distribution:
 label
1    433512
0     66488
Name: count, dtype: int64
Saved: /kaggle/working/submission_full.csv


,ID,label
0,0,0
1,2,0
2,5,1
3,6,0
4,7,0
